# Uncertainty Quantification Exercises (3.3 a–e)

## Problem definition
We have wind conditions such that the single-sample wind speed $S$ can be characterized by a stationary random process, where the 10-minute  $\langle S\rangle = 10.5\,$m/s with standard deviation $\sigma_S = 0.9\,$m/s. 

We have a wind turbine power curve described by
$P(S) = \frac{1}{2}C_p\rho A S^3$ up to rated speed $S_\textrm{rated}$, above which the power is constant at $P=P_\textrm{rated}$; here $A$ is the swept area.

We consider a **7 MW** turbine with rotor diameter $D=$140m, $C_p=$0.45, and air density $\rho=1.2\,$kg/m$^3$.

### Task 3.3a
- What is $S_\textrm{rated}$, given the $\{C_p,\rho,D\}$ and $P_\textrm{rated}$ that we consider here?
- Compute the uncertainty in the instantaneous power output ($U_P$):
    * using the GUM method; and
    * using Monte Carlo ['MC'] simulation. 
- How do the GUM and MC results compare/differ? 
- What about other wind speeds, e.g., if  $\langle S\rangle=15\,$m/s, does everything work as expected?

#### Hints /reminders
   The GUM method defines uncertainty propagation assuming linear dependence between the inputs and outputs near a point of interest (typically the expected value); it provides a linearized approximation to the uncertainty.  Thus <u> GUM is valid only close to the specific point where the linear approximation is made </u>.  Note that _"close"_ can be defined quantitatively, in nondimensional terms... 
For a value $x$ with $\Delta x\equiv |x-x_0|$,  _"close"_ would mean $\Delta x\ll x_0$, or nondimensionally $\frac{\Delta x}{x_0} \ll 1$.

For a function $y = g(x)$, the mean and variance of $y(x)$ close to a point $x_0$ can be estimated in terms of the mean $E(x)$ and variance $V(x)$ by

$\qquad E(y) \approx g(E(x))$ , that is $\overline{y}\approx g(\overline{x})$ ; 

$\qquad V(y) \approx \left(\frac{\mathrm{d}y}{\mathrm{d}x}\big|_{x_0}\right)^2 V(x)$.

In [2]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
Cp =         # power coefficient [unitless]
rho =        # density in units of kg/m^3
D =          # rotor diameter in units of m
A =          # calculate rotor area
unitsP = 1e6   # let's use MW for units of power; recall that rho*A*S^3 gives units of W
P_rated = 7.   # 7MW, in units of MW
Smean =      # mean windspeed in units of m/s
Sstd =       # std.dev. of windspeed in units of m/s
# make power curve, with units of MW ;  "np.minimum" allows transition from increasing power to P=P_rated
PowerFunc = lambda S: np.minimum(P_rated, (insert_equation_for_power_in_terms_of_S)/unitsP) 

In [ ]:
# Check - did we get the right power curve? 
S = np.arange(1,25,0.1)
plt.figure(figsize=(3, 2)); plt.xlabel('$U$ [m/s]'); plt.ylabel('$P$ [MW]'); 
plt.plot(S,PowerFunc(S));

In [ ]:
# calculate speed where rated power starts (Vrated) 
Vrated = ...
print("V_rated=",np.round(Vrated,1), "m/s")

In [ ]:
# Carry out a Monte Carlo simulation

NMC = 10**...    # number of samples in Monte-Carlo
# Generate NMC random windspeed samples, normally-distributed with mean=Smean and std.dev.=Sstd
Srand = ...  + np.random.randn(NMC) *  ...

Prand = PowerFunc(Srand) # Python "broadcasts" the array, to match dimensions of Srand first

PmeanMC = np.mean(Prand)
PstdMC = np.std(Prand)

# Compute empirical PDF (same procedure as in exercise 1.1)
Pbin_min=...; Pbin_max=...; n_Pbins=        # pick bins appropriate for P_rated, consider NMC
Pbinedges = np.linspace(Pbin_min, Pbin_max, n_Pbins)
Pbins = Pbinedges[0:-1] + np.diff(Pbinedges)/2
Phist = np.histogram(Prand, bins=Pbinedges)
rectangle_rule = lambda x,y: np.sum(np.diff(x[0:y.shape[0]+1])*y)
PpdfMC = Phist[0]/rectangle_rule(Phist[1],Phist[0])

In [ ]:
# GUM estimate
dPdS = lambda S: (some_Function_of_Cp,A,rho,S)/unitsP # derivative of power with respect to wind speed
# above insert the derivative above within parenthesis, replacing "some_Function_of_Cp,A,rho,S"
PmeanGUM = PowerFunc(Smean)
PstdGUM = dPdS(Smean)*Sstd
PpdfGUM = stats.norm.pdf(Pbins,PmeanGUM,PstdGUM)

In [ ]:
print('Mean power according to GUM: ' + str(round(PmeanGUM,3)) + ' MW')
print('Mean power according to MC: ' + str(round(PmeanMC,3)) + ' MW')
print('St.dev. of power via GUM: '+str(round(PstdGUM,3))+' MW ('+str(round(100*PstdGUM/PmeanGUM,1))+'%)')
print('St.dev. of power via MC: '+str(round(PstdMC,3))+' MW ('+str(round(100*PstdMC/PmeanMC,1))+'%)')

In [ ]:
fig0,axs0 = plt.subplots(1,2,figsize = (11,4))
axs0[0].hist(Prand,100)
axs0[1].plot(Pbins/P_rated,PpdfMC)
axs0[1].plot(Pbins/P_rated,PpdfGUM)
axs0[0].set_ylabel('Number of occurences',fontsize = 12); axs0[0].set_xlabel('power/P_rated',fontsize = 12)
axs0[0].set_title('Histogram of Monte Carlo outcomes')
axs0[1].set_xlabel('power/P_rated',fontsize = 12); axs0[1].set_ylabel('Probability density',fontsize = 12)
axs0[1].set_title('Probability density functions')
axs0[1].legend(('Monte Carlo','GUM'),loc = 'upper left')
plt.show()

now repeat the above with higher speed to check what will happen...

### Task 3.3b:
Now assume that we are collecting random samples, with each consisting of 600 independent measurements of the variables $S$ and $P$. 
- What is the uncertainty in the sample-mean of power output? 
- Does the law of large numbers, and the central limit theorem, apply?


In [ ]:
# Task 3.3(b): Carry out a MC using nMC 'samples', where each sample is of size 600 (normally-distributed)
nMC = # a big number
SMC = # use np.random.randn(neach,nMC) along with Smean and Sstd, with 600 independent obs for each MC sim
PMC = PowerFunc(SMC)
PmeansMC = # mean of PMC for each sample
PuncMC = # use PmeansMC

print('Standard deviation of the sample mean, for sample size 600: sigma_P|N = ' + str(round(PuncMC,3))+' MW, or ' \
      + str(round(student:express_Punc_as_percentage_here,1))+'%' )
print('Sampling uncertainty based on MC with sample size 1 & division by sqrt(N): ' \
      + str(round(PuncMC/np.sqrt(600),3))+' MW, or '+ str(round(student_reexpress_as_percentage,2))+'%' )

plt.xlabel(r'$\langle P\rangle_{N=600}$ [MW]'); plt.ylabel('pdf of '+r'$\langle P\rangle_{N=600}$ [MW$^{-1}$]'); 
plt.hist(PmeansMC,100);   plt.show()

### task 3.3(c):  now try random $C_p$ uncertainty also, independent of $S$

Assume 20% uncertainty in $C_p$, that is $\langle C_p^2 \rangle^{1/2} = 0.2\langle C_p\rangle$.

In [ ]:
meanCp = 0.45;  stdCp=0.2*meanCp
NMC = 100000
Srand = # random S from a Normal distribution
CpRand = # random Cp from a Normal distribution

In [ ]:
PowerFunc2 = lambda S,Cp: np.minimum(P_rated, Cp*0.5*rho*A*S**3/unitsP)  # now 2 inputs to powerCurve
Prand2 = PowerFunc2(Srand,CpRand)

In [ ]:
# MC estimate
Phist2 = np.histogram(Prand2, bins=Pbinedges)
PpdfMC2 = Phist2[0]/rectangle_rule(Phist2[1],Phist2[0])

In [ ]:
# GUM estimate
dPdCp = lambda S: 0.5*A*rho*S**3/unitsP # derivative of power with respect to Cp

PmeanGUM2 = PowerFunc2(Smean,meanCp)
PstdGUM2 = # combine the estimated uncertainties
PpdfGUM2 = stats.norm.pdf(Pbins,PmeanGUM2,PstdGUM2)

In [ ]:
plt.plot(Pbins,PpdfMC,Pbins,PpdfGUM2,Pbins,PpdfMC2);  # plot (a,c) together

### Task (d): now try random $C_p$ that is somewhat anti-correlated with $S$ 
assume correlation coefficient $\rho_{c_p S} = -0.5$, that is $\langle C_p S\rangle = -0.5 \sigma_{c_p}\sigma_S$

In [ ]:
meanCp = 0.45;  stdCp=0.2*meanCp
meansSandCp = np.array([Smean,meanCp])
covSCp = np.array([[variance_of_S,covariance_of_S_&_Cp],[covariance_of_S_&_Cp,variance_of_Cp]])

In [ ]:
NMC =   # number of Monte Carlo "sims"
SCpMC = np.random.multivariate_normal(meansSandCp,covSCp,size=NMC)

In [ ]:
PWRpartD = PowerFunc2( ... , ... )

In [ ]:
# MC estimate
PhistD = np.histogram(PWRpartD, bins=Pbinedges)
PpdfMCpartD = PhistD[0]/rectangle_rule(PhistD[1],PhistD[0])

In [ ]:
# GUM estimate
PmeanGUMd = PowerFunc2(Smean,meanCp)
PstdGUMd = np.sqrt((dPdS(Smean)*Sstd)**2 + (dPdCp(Smean)*stdCp)**2 - fill_in_this_cross-term )
PpdfGUMd = stats.norm.pdf(Pbins,PmeanGUMd,PstdGUMd)

In [ ]:
plt.plot(Pbins,PpdfMCpartD,Pbins,PpdfGUMd,Pbins,PpdfMC2);   # plot (d) and MC from (c)

### Task (e): redo again, now with    $C_p \propto S^{-1/2}$ 

In [ ]:
## repeat like above, but now with covariance matrix according to the {Cp,S} relationship  ...

In [ ]:
CpRandFromS = ...  # hint: normalize sqrt(S) and multiply it by mean of Cp

In [ ]:
## check the correlation matrix for Cp and S here... what should corr(Cp,S) be ?
print(np.corrcoef(CpRandFromS,SpartD))

In [ ]:
## repeat like part d, but now with covariance matrix (or more simply the correlation coeff. verified above)
